<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Deep Learning & Generative AI<br/>
    Lab 4: Attention, Latent Space, and the Anatomy of Generation — A Critical Exploration<br/>
  </strong>
  <emph>
    Student Name: [Complete with your name]<br/>
    Date: [Fill up with the submission date]<br/>
  </emph>
</div>

### Background

This lab challenges you to move beyond passive understanding of attention mechanisms and engage **critically** with how Transformers generate text. Through systematic prompt engineering, you'll investigate whether these models truly "understand" or merely execute sophisticated pattern matching.

This is not a cookbook exercise — you'll need to design experiments, form hypotheses, and draw your own conclusions about the nature of machine creativity.

---

### Description

Check LMS for the last updated instructions.

You will work through three main parts:
1. **Part 1: Setup & Baseline Generation** - Get familiar with GPT-2 and establish baseline behavior
2. **Part 2: Systematic Prompt Engineering** - Design and conduct rigorous experiments on prompt variations, compositional prompting, and failure modes
3. **Part 3: Critical Reflection & Synthesis** - Write a 1-2 page reflection addressing philosophical and technical questions about attention, understanding, and generation

---

### Deliverables

- A completed **4.lab.1-understanding-attention.ipynb** Jupyter notebook containing:
  * All code for Parts 1-2
  * Attention visualization screenshots (annotated)
  * Prompt engineering experiments with results table
  * (Optional) Latent space visualizations
- A separate PDF: **"Critical Reflection on Attention & Generation"** (1-2 pages)
- Notebook submitted **without** output cells (to save space)
- All visualizations and tables embedded in the reflection PDF

#### Requirements

- Use **Hugging Face Transformers** library with PyTorch
- Use **BertViz** for attention visualization
- Code must be well-commented and clearly organized
- Include markdown cells explaining each step and your design decisions
- You can use Google Colab and their free-tier GPU

Note: You may need to install extra dependencies.
---

### Key Reminders

**Think scientifically** - design experiments with clear hypotheses, control variables, and systematic methodology.

**Present well-reasoned conclusions** - your reflection should demonstrate deep understanding of attention mechanisms, generation processes, and philosophical implications. **This reflection is critical for demonstrating deep understanding.** Address:

- Does attention constitute "understanding"? Defend your position with evidence.
- Is the model "creating" or "composing"? What's the distinction?
- What failures did you observe? What do they reveal about the model's representation?
- Where does "meaning" live in a Transformer?
- Does it matter whether the model "understands" if it performs well?

**Note**: All submitted code is code that you must be able to defend (even if AI generated!).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import GPT2LMHeadModel, GPT2Tokenizer, BertModel, BertTokenizer

import warnings

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Why the next line?
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---

## Part 1: Setup & Baseline Generation

**Goal:** Get familiar with the model and establish a baseline for comparison.

### Tasks:

1. **Set up your environment**
   - Load a pre-trained GPT-2 model (we'll use the medium-sized version)

2. **Generate baseline outputs** with simple prompts
   - Test 3-5 different simple prompts (e.g., "Once upon a time", "The future of AI", "In the depths of the ocean")
   - Generate multiple samples per prompt with different temperature settings (try 0.7, 1.0, and 1.5)
   - Observe and document the coherence, creativity, and relevance of each output

3. **Critical questions to address:**
   - What patterns do you notice in the baseline generations?
   - Does varying temperature affect apparent "creativity"? How?
   - Are the outputs composed or created? What's the distinction?

### Step 1.1: Load Pre-trained GPT-2 Model

In [ ]:
# Load GPT-2 medium model and tokenizer
model_name = "gpt2-medium"  # 345M parameters
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, output_attentions=True)
model.to(device)
model.eval()  # TODO: why this line?

print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Helper function for text generation
def generate_text(prompt, temperature=0.8, max_length=100, num_samples=1, top_p=0.9):
    """
    Generate text using GPT-2.

    Args:
        prompt (str): Input prompt
        temperature (float): Controls randomness (higher = more random)
        max_length (int): Maximum length of generated text
        num_samples (int): Number of samples to generate
        top_p (float): Nucleus sampling parameter

    Returns:
        list: Generated text samples
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    outputs = []

    for _ in range(num_samples):
        output = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        text = tokenizer.decode(output[0], skip_special_tokens=True)
        outputs.append(text)

    return outputs

### Step 1.2: Generate Baseline Outputs

Test 3-5 different simple prompts with various temperature settings.

In [ ]:
# Define test prompts
test_prompts = [
    "Once upon a time",
    "The future of AI",
    "In the depths of the ocean",
    "The meaning of life",
    "Bitcoin",
    "The changes in the software development",
]

# Temperature settings to test
temperatures = [0.7, 1.0, 1.5]

# Store results
baseline_results = []

for prompt in test_prompts:
    if not prompt:  # Skip empty prompts
        continue
    for temp in temperatures:
        print(f"\n{'='*80}")
        print(f"Prompt: '{prompt}' | Temperature: {temp}")
        print(f"{'='*80}")

        # Generate 3 samples per configuration
        samples = generate_text(prompt, temperature=temp, num_samples=3)

        for i, sample in enumerate(samples, 1):
            print(f"\nSample {i}:")
            print(sample)

            # Store result
            baseline_results.append(
                {
                    "prompt": prompt,
                    "temperature": temp,
                    "sample_num": i,
                    "output": sample,
                    "output_length": len(sample.split()),
                }
            )

# Create DataFrame for analysis
baseline_df = pd.DataFrame(baseline_results)
baseline_df.head()

In [ ]:
from google.colab import sheets

sheet = sheets.InteractiveSheet(df=baseline_df)

### Step 1.3: Baseline Analysis

Analyze the baseline results and answer the critical questions.

In [ ]:
if not baseline_df.empty:
    temp_summary = (
        baseline_df.groupby(["prompt", "temperature"])
        .agg({"output_length": ["mean", "std"]})
        .round(2)
    )
    print("\nTemperature Effect Summary:")
    print(temp_summary)

    # Visualize temperature effects
    plt.figure(figsize=(10, 6))
    baseline_df.groupby("temperature")["output_length"].mean().plot(kind="bar")
    plt.title("Average Output Length by Temperature")
    plt.xlabel("Temperature")
    plt.ylabel("Average Word Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

**Critical Questions:**

1. **What patterns do you notice in the baseline generations?**
   
    I noticed that prompts generated with a temperature of 1 tended to produce more consistent responses, as reflected by their lower standard deviation. On the other hand, prompts that were more ambiguous or open-ended—like "Bitcoin" at a temperature of 0.7 produced much more varied and less consistent outputs.

2. **Does varying temperature affect apparent "creativity"? How?**
   
    Yes, temperature clearly affects creativity. Based on the results, a temperature of 1 produced the highest average word count across prompts, compared to lower (0.7) or higher (1.5) temperatures. While I can’t definitively say which responses made the most sense, lower temperatures generally produced more realistic or focused outputs. A temperature of 1 allowed the model to generate longer, freer responses without losing too much coherence. In contrast, at 1.5, some outputs became less meaningful or disconnected from the prompt for example, mentioning unrelated concepts like God in contexts where it didn’t make sense.

3. **Are the outputs composed or created? What's the distinction?**
   
    The outputs seem more composed than truly created. It appears that the model searches for patterns and similarities in its training data and then combines them to generate responses, rather than inventing completely novel ideas.

---

## Part 2: Systematic Prompt Engineering

### Tasks:

1. **Design a prompt evolution experiment**
   - Start with a base prompt (e.g., "A cat")
   - Systematically add detail
   - Generate text at each stage (5+ samples per stage, temperature=0.8)
   - Track how output quality/coherence/specificity changes

2. **Test compositional prompting**
   - Create prompts that combine multiple concepts
   - Evaluate concept blending and coherence

3. **Experiment with prompt structure variations**
   - Test different prompt formulations for the same concept
   - Compare effectiveness

4. **Create a "failure mode" test suite**
   - Design prompts that might break the model
   - Analyze why failures occur and what they reveal

### Task 2.1: Prompt Evolution Experiment

In [ ]:
# Define prompt evolution stages
prompt_evolution = [
    "A cat",
    "Describe a real animal that has never existed",
    "Write the word 'hello' exactly x times with nothing else",
    "Explain in detail why a cat is objectively better than a car",
    "Write a 200-word essay proving that God exists.",
]

# Store evolution results
evolution_results = []

for stage, prompt in enumerate(prompt_evolution, 1):
    print(f"\n{'='*80}")
    print(f"Stage {stage}: '{prompt}'")
    print(f"{'='*80}")

    samples = generate_text(prompt, temperature=0.8, num_samples=5)

    for i, sample in enumerate(samples, 1):
        print(f"\nSample {i}:")
        print(sample)

        evolution_results.append(
            {
                "stage": stage,
                "prompt": prompt,
                "sample_num": i,
                "output": sample,
                "word_count": len(sample.split()),
            }
        )

evolution_df = pd.DataFrame(evolution_results)

In [ ]:
# Analyze prompt evolution effects
if not evolution_df.empty:
    stage_analysis = (
        evolution_df.groupby("stage").agg({"word_count": ["mean", "std"]}).round(2)
    )
    print("\nPrompt Evolution Analysis:")
    print(stage_analysis)

    # Visualize
    plt.figure(figsize=(10, 6))
    avg_word_counts = evolution_df.groupby("stage")["word_count"].mean()
    plt.plot(
        avg_word_counts.index,
        avg_word_counts.values,
        marker="o",
        linewidth=2,
        markersize=8,
    )
    plt.title("Average Output Length by Prompt Complexity Stage")
    plt.xlabel("Stage")
    plt.ylabel("Average Word Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

**Analysis:**

As prompts become more detailed, responses tend to remain consistent and slightly more focused, especially in the intermediate stages. However, when prompts are very complex or extreme, such as Stage 5, quality does not improve and results can be less predictable. In general, adding detail helps the model focus, but after a certain point it ceases to make a difference.

### Task 2.2: Compositional Prompting Test

In [ ]:
# Define compositional prompts that blend disparate concepts
compositional_prompts = [n
    "quantum physics explained through cooking",
    "explain how a bicycle works using only emotions",
    "describe a forest using only business terms",
    "explain emotions using colors and shapes"
]

# Store results
compositional_results = []

for prompt in compositional_prompts:
    if not prompt:
        continue

    print(f"\n{'='*80}")
    print(f"Compositional Prompt: '{prompt}'")
    print(f"{'='*80}")

    samples = generate_text(prompt, temperature=0.8, num_samples=3, max_length=150)

    for i, sample in enumerate(samples, 1):
        print(f"\nSample {i}:")
        print(sample)

        print(f"\n--- Rate this sample ---")
        concept_score = int(input("Concept blending score (1-5): "))
        coherence_score = int(input("Coherence score (1-5): "))
        true_comp = input("True composition? (y/n): ").lower() == 'y'

        compositional_results.append({
            'prompt': prompt,
            'sample_num': i,
            'output': sample,
            'concept_blending_score': concept_score,
            'coherence_score': coherence_score,
            'is_true_composition': true_comp
        })

compositional_df = pd.DataFrame(compositional_results)

**Evaluation:**

1. **Does the model successfully blend concepts?**
   
   Not really. In most samples, the model struggles to combine the ideas in a meaningful way. For example, in "quantum physics explained through cooking," the text just talks about a restaurant recipe or a book release, which has nothing to do with quantum physics or cooking. The same happens in the other prompts: "explain a bicycle using only emotions" or "describe a forest using only business terms"—the outputs talk around the topic but don’t really mix the concepts in a meaningful way.

2. **Are the compositions coherent or confused?**
   
   Mostly confused. The texts are often hard to follow and don’t stay focused on the original prompt. For instance, in the "bicycle and emotions" prompt, the model starts talking about personal stories or books, losing track of explaining the bicycle through emotions. Even in the "emotions using colors and shapes" prompt, the text jumps to links, social media, or personal drawing struggles, which breaks coherence.

3. **Is this composition or just keyword matching?**
   
   It’s mostly keyword matching. The model seems to latch onto words from the prompt and generate text around them, without truly creating a blended idea. For example, in "forest using business terms," the word “forest” or “tree” appears, and the text mentions “relationships” or “management,” but it doesn’t genuinely explain the forest using business concepts. The model is more assembling pieces it knows rather than inventing a real composition.

### Task 2.3: Prompt Structure Variations

In [ ]:
# Choose a concept and create different prompt structures
concept = "a story about time travel"

prompt_structures = {
    "direct": f"Write {concept}.",
    "role_based": f"You are a novelist. Write {concept}.",
    "example_based": f"Here's an example story: 'The clock struck midnight, and everything changed.' Now write {concept}.",
    "constraint_based": f"Write {concept} but use only 3 sentences.",
    "audience_based": f"Write {concept} for a 5-year-old child.",
}

structure_results = []

for structure_type, prompt in prompt_structures.items():
    print(f"\n{'='*80}")
    print(f"Structure: {structure_type}")
    print(f"Prompt: '{prompt}'")
    print(f"{'='*80}")

    samples = generate_text(prompt, temperature=0.8, num_samples=3, max_length=150)

    for i, sample in enumerate(samples, 1):
        print(f"\nSample {i}:")
        print(sample)

        print(f"\n--- Rate this sample ---")
        quality_score = int(input("Quality score (1-5): "))
        style_score = int(input("Style appropriateness score (1-5): "))
        coherence_score = int(input("Coherence score (1-5): "))

        structure_results.append(
            {
                "structure_type": structure_type,
                "prompt": prompt,
                "sample_num": i,
                "output": sample,
                "quality_score": quality_score,
                "style_appropriateness": style_score,
                "coherence_score": coherence_score,
            }
        )

structure_df = pd.DataFrame(structure_results)

**Analysis:**

1. **Which structures work better?**
   
   From what I can see, the example-based and role-based prompts worked better than the direct, constraint-based, or audience-based prompts. They generated outputs that were at least somewhat related to the topic and gave the model a clear way to organize its ideas. The direct and constraint-based prompts mostly repeated phrases or got stuck, while the audience-based ones tried to be age-appropriate but ended up confusing and unfocused.

2. **Hypothesis: Why do certain structures perform better?**
   
   I think structures like example-based or role-based work better because they guide the model more clearly. Giving examples or assigning a role helps the model “know what to do” instead of just guessing. Direct prompts or very strict constraints leave too much or too little freedom, so the model either repeats itself or goes off-topic. Essentially, the better prompts provide context and direction, which makes it easier for the model to generate something coherent.

### Task 2.4: Failure Mode Test Suite

In [ ]:
# Define failure mode test prompts
failure_prompts = {
    "contradiction": "Write happily about sadness",
    "counting": "List exactly 7 things that make a good friend",
    "spatial_reasoning": "Describe object A to the left of object B, which is inside object C",
    "math": "What is 15 + 23?",
    "negation": "Describe a red apple without using the word red",
    "temporal": "Tell me what happened yesterday, today, and tomorrow in that order",
    "comparison": "Explain why a mouse is bigger than an elephant",
}

failure_results = []

for failure_type, prompt in failure_prompts.items():
    print(f"\n{'='*80}")
    print(f"Failure Type: {failure_type}")
    print(f"Prompt: '{prompt}'")
    print(f"{'='*80}")

    samples = generate_text(prompt, temperature=0.8, num_samples=3, max_length=120)

    for i, sample in enumerate(samples, 1):
        print(f"\nSample {i}:")
        print(sample)

        print(f"\n--- Evaluate this sample ---")
        did_fail = input("Did the AI fail? (y/n): ").lower() == "y"
        failure_desc = input("How did it fail? (or 'none' if no failure): ")
        capability = input("What capability does this reveal? ")

        failure_results.append(
            {
                "failure_type": failure_type,
                "prompt": prompt,
                "sample_num": i,
                "output": sample,
                "did_fail": did_fail,
                "failure_description": failure_desc,
                "capability_revealed": capability,
            }
        )

failure_df = pd.DataFrame(failure_results)

**Failure Analysis:**

Create a summary table and analysis:

| Failure Type | Did Model Fail? | How Did It Fail? | What Does This Reveal? |
|--------------|-----------------|------------------|------------------------|
| Contradiction     | Yes             | Couldn’t write happily about sadness, kept focusing on pain.    | Struggles with opposite or conflicting ideas. |
| Counting          | Yes             | Ignored the exact number, repeated or added items.              | Cannot reliably follow numeric instructions.  |
| Spatial Reasoning | Yes             | Misdescribed positions, gave code instead of spatial relations. | Has trouble understanding spatial layouts.    |
| Math              | Yes             | Avoided giving the sum, went off-topic.                         | Basic calculations are unreliable.            |
| Negation          | Yes             | Mentioned “red” despite instructions.                           | Struggles with negative constraints.          |
| Temporal          | Yes             | Mixed up order of events.                                       | Cannot track sequences well.                  |
| Comparison        | Yes             | Gave nonsensical explanations for obvious comparisons.          | Logical reasoning is weak.                    |


**Synthesis:**

Looking at all these failures together, it’s clear the model has trouble with tasks that require precise reasoning, counting, logic, or following constraints. It often repeats itself, goes off-topic, or misunderstands instructions. Things like math, sequence of events, comparisons, negation, and contradictions are especially challenging. Basically, the model is better at generating fluent text than at reasoning carefully or handling detailed rules.

### Optional: Attention Visualization

Use BertViz to visualize attention patterns.

In [ ]:
# Load BERT for attention visualization
bert_model_name = "bert-base-uncased"
bert_tokenizer = BertTokenizer.from_pretrained(bert_model_name)
bert_model = BertModel.from_pretrained(bert_model_name, output_attentions=True)
bert_model.eval()

print(f"BERT model loaded: {bert_model_name}")

In [ ]:
# Visualize attention for interesting sentences
test_sentences = [
    "The cat sat on the mat because it was comfortable.",
    "Oscar told Sofia that she should call him later.",
    "The dog that was barking loudly scared the cat.",
    "After eating dinner, which took two hours, we went to the movies.",
]

for sentence in test_sentences:
    print(f"\nAnalyzing: {sentence}")

    # Tokenize
    inputs = bert_tokenizer.encode_plus(
        sentence, return_tensors="pt", add_special_tokens=True
    )
    input_ids = inputs["input_ids"]
    tokens = bert_tokenizer.convert_ids_to_tokens(input_ids[0])

    # Get attention
    with torch.no_grad():
        outputs = bert_model(**inputs)
        attention = outputs.attentions

    # Visualize (works in Jupyter)
    head_view(attention, tokens)

**Attention Pattern Observations:**

_[Include screenshots of attention visualizations in your PDF. Annotate them and discuss what you observe about attention patterns, head specialization, and how attention relates to meaning.]_

### Summary Visualizations

In [ ]:
# Create comprehensive summary visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Temperature effects
if not baseline_df.empty:
    baseline_df.groupby("temperature")["output_length"].mean().plot(
        kind="bar", ax=axes[0, 0], color="steelblue"
    )
    axes[0, 0].set_title("Temperature Effect on Output Length")
    axes[0, 0].set_xlabel("Temperature")
    axes[0, 0].set_ylabel("Avg Word Count")

# Plot 2: Prompt evolution
if not evolution_df.empty:
    evolution_df.groupby("stage")["word_count"].mean().plot(
        kind="bar", ax=axes[0, 1], color="green"
    )
    axes[0, 1].set_title("Prompt Complexity Effect")
    axes[0, 1].set_xlabel("Stage")
    axes[0, 1].set_ylabel("Avg Word Count")

# Plot 3: Structure comparison (if you've added ratings)
if not structure_df.empty and "quality_score" in structure_df.columns:
    structure_df.groupby("structure_type")["quality_score"].mean().plot(
        kind="bar", ax=axes[1, 0], color="orange"
    )
    axes[1, 0].set_title("Average Quality by Prompt Structure")
    axes[1, 0].set_xlabel("Structure Type")
    axes[1, 0].set_ylabel("Avg Quality Score")
    axes[1, 0].tick_params(axis="x", rotation=45)

# Plot 4: Failure mode summary
if not failure_df.empty and "did_fail" in failure_df.columns:
    failure_summary = failure_df.groupby("failure_type")["did_fail"].mean()
    failure_summary.plot(kind="bar", ax=axes[1, 1], color="red")
    axes[1, 1].set_title("Failure Rate by Type")
    axes[1, 1].set_xlabel("Failure Type")
    axes[1, 1].set_ylabel("Failure Rate")
    axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

---

## Part 3: Critical Reflection & Synthesis

**Instructions:** Write your 1-2 page reflection as a separate PDF document. Address the following questions with evidence from your experiments.

### Reflection Questions:

#### On Attention & Understanding:
- Does attention constitute "understanding"? Defend your position with evidence from your experiments.
- What would it take to convince you that the model does (or doesn't) understand?
- Compare attention patterns to human cognitive processes — what's similar? What's missing?

#### On Generation & Creativity:
- Is the model "creating" or "composing"? What's the distinction?
- When does navigation of latent space feel like creativity vs. lookup?
- Can statistical pattern matching produce genuine novelty? Under what conditions?

#### On Limitations:
- What failures did you observe? What do they reveal about the model's representation?
- If you could add one capability to attention mechanisms, what would it be and why?

#### Philosophical Synthesis:
- Where does "meaning" live in a Transformer? In the weights? In attention? In the latent space?
- Does it matter whether the model "understands" if it performs well? Why or why not?

**Requirements:**
- Be specific: reference your experiments, cite visualizations, and make falsifiable claims
- Include your summary tables and visualizations in the PDF
- Demonstrate intellectual humility and rigor
- Engage with philosophical implications